Name

Labpartner(s)

In [1]:
#import statements go here
import numpy as np
import pandas as pd

# Class 10.2

Today we are having a guest lecture by Kesse Asante on species distribution modeling in R. 

**W.1** Practice python #13

The Fibonnaci seqence is a sequence of numbers where the next number in the sequence is the sum of the previous two numbers in the sequence. The sequence looks like this: 1, 1, 2, 3, 5, 8, 13, …

Write a program that asks the user how many Fibonnaci numbers to generate and then generates them. Take this opportunity to think about how you can use functions. Make sure to ask the user to enter the number of numbers in the sequence to generate.

In [2]:
n = input('Sequence length:')

# Fibonnaci sequence starts with 1, 1
def fib(n):
    a = 1
    b = 1
    # For loop to generate the numbers; use '_' to run loop n times
    for c in range(int(n)):
        print(a, end = ' ')
        a, b = b, a + b

fib(n)

1 1 2 3 5 8 13 21 34 55 

# Lecture 10.2

### Agenda: Asante will have you install some R, then he will walk you through some species distribution modeling

**Please see the pdf document for Lab 10.2 and get the downloads going.**

### Questions

# Lab 10.2

**E.0** Complete the previous labs if you are behind.

**E.1** Asante will give you some exercises to do in R

Below is the R script I used for the class assignment.

In [ ]:
####### Class excersie
#shapfile for land
landmass <- vect('Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Landmass/landmass.shp')

# plot map
plot(landmass, col = '#D4D4D4', border = 'darkgrey', axes = TRUE)

# presence data
presence <- read.csv('Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Corals/occurrenceRecords.csv')
# turn df to spatial
presence <- vect(presence, geom = c('Lon','Lat'))

# plot records
plot(landmass, main = 'Species distribution records', col = '#D4D4D4', border = 'darkgrey', axes = TRUE)
plot(presence, col = 'aquamarine4', pch = 16, cex = 0.5, add = TRUE)

files <- c ('Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Present Surface OceanTemperature LtMin.tif', # Surface temp min
            'Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Present Surface OceanTemperature LtMax.tif', # surface temp max
            "Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMax.tif", # Temperature Max
            "Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Present Benthic OceanTemperature LtMin.tif", # Temperature Min
            "Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Present Benthic pH Mean.tif", # pH
            "Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Present Benthic Salinity Mean.tif" # Salinity 
)

#produce and plot raster layers
environmentalLayers <- rast(files)
names(environmentalLayers) <- c('Present_Surface_OceanTemperature_LtMin.tif','Present_Surface_OceanTemperature_LtMax.tif','Present_Benthic_OceanTemperature_LtMax','Present_Surface_OceanTemperature_LtMin.tif','Present_Benthic_pH_Mean','Present_Benthic_Salinity_Mean')
plot(environmentalLayers)

# extent for Tropical America and Caribbean 
extent <- ext(-100, -55, 0, 32)

presence_extent <- crop(presence, extent)
plot(presence_extent)

#develop a study region layer
myStudyRegion <- studyRegion(rasterLayers=environmentalLayers,
                             presences=presence_extent, # presence records
                             distanceThreshold=200000, # distance from records in meters
                             bathymetryLayer = "Classes/OCS 7001 Scientific Computing (Fall 2026)/Sci_Comp26/Data/Raster_data/Bathymetry.tif", 
                             depthPref=c(0,-60))
#inspect the study region layer
plot(myStudyRegion, main="Study region", col="#979797", axes=TRUE)

#crop the environmental layers to the extent of the study region
environmentalLayers <- crop(environmentalLayers, myStudyRegion)
#mask the environmental layers with the study region
environmentalLayers <- mask(environmentalLayers, myStudyRegion)
#plot the masked environmental layers
plot(environmentalLayers, axes = TRUE)

# generate pseudo-absence data
absence <- pseudoAbsences(rasterLayers = environmentalLayers, presences = presence_extent, n = 2000)

# crop landmass to study region
landmass_crop <- crop(landmass, myStudyRegion)

plot(landmass_crop, main = 'Tropical and Caribbean Coarls', col = '#D4D4D4', border = 'darkgrey', axes = TRUE)
plot(absence, col = 'violetred4', pch = 16, cex = 0.5, add = TRUE)
plot(presence, col = 'aquamarine4', pch = 16, cex = 0.5, add = TRUE)
legend('topright', legend = c('Absent','Present'), col = c('violetred4','aquamarine4'),
       pch = 16, pt.cex = 1, cex = 0.75, bty = 'n', inset = c(0.15, 0.05))

# combine presence/absence into single object for model reference
presence_extent$PA <- 1
absence$PA <- 0
records <- rbind(presence_extent, absence)

# plot to make sure it worked
plot(landmass_crop, main = 'Tropical and Caribbean Coarls', col = '#D4D4D4', border = 'darkgrey', axes = TRUE)
plot(records, y = 'PA', col = c('violetred4', 'aquamarine4'), pch = 16, cex = 0.5, add = TRUE)

#extract environmental values and make an object with all information to model, model evaluation with cross validation
modelData <- generateModelData(records, envConditions = environmentalLayers, method = 'blocks', kFolds = 10)
# inspect data partitioning
modelData$plotDatasets

#fit a binomial GLM to the dataset
model <- trainModel(modelData, algorithm = "brt")
#predict the distribution with the model over a raster stack
prediction <- predictModel(model = model, newData = environmentalLayers)
#map the predicted distribution
plot(landmass_crop, main = "Predicted species distribution", col = '#D4D4D4', border = 'darkgrey', axes = TRUE)
plot(prediction$rasterLayer, col = rev(topo.colors(100)), add = TRUE)

**E.2** Make notes for yourself on progamming tecniques and commands you learned in the this week, including examples, comments and explainitory text. You can do this here or in a separate notebook that you link to here. Basically, you are making a cheat sheet for yourself.